# Systems

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/tutorial/core/01_systems.ipynb)

The `System` contract and its family: continuous, discrete, sampled, and the diagrams that compose them.

In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

## 1. A system is its equations

Every model in minilink is a `System`: a set of named ports, a state of dimension `n`, a
nominal initial state `x0`, a parameter dictionary `params`, and its equations. A
`DynamicSystem` is the continuous kind:

$$\dot x = f(x, u, t; p), \qquad y = h(x, u, t; p)$$

The constructor creates the standard ports (an input `u` of size `input_dim`, an output `y`
of size `output_dim`). `f` and `h` read the parameters from `params`, falling back to
`self.params`, and build their arrays through `array_module(x)`, so the same equation runs
on NumPy and on JAX arrays. Here is a mass-spring-damper, $m \ddot q = F - k q - c \dot q$,
that measures its position.

In [ ]:
import numpy as np

from minilink import DynamicSystem
from minilink.core.backends import array_module


class MassSpringDamper(DynamicSystem):
    def __init__(self):
        super().__init__(2, input_dim=1, output_dim=1)
        self.name = "Mass-spring-damper"
        self.params = {"m": 1.0, "k": 4.0, "c": 0.4}
        self.x0 = np.array([1.0, 0.0])
        self.state.labels = ["q", "dq"]

    def f(self, x, u, t=0.0, params=None):
        params = self.params if params is None else params
        xp = array_module(x)

        m = params["m"]
        k = params["k"]
        c = params["c"]

        q = x[0]
        dq = x[1]
        F = u[0]

        # m q̈ = F − k q − c q̇
        ddq = (F - k * q - c * dq) / m
        dx = xp.stack([dq, ddq])

        return dx

    def h(self, x, u, t=0.0, params=None):
        y = x[0:1]

        return y

The object is a description: printing it lists its dimensions and ports. `f` is a plain
function of its arguments. An explicit `params` dictionary overrides the stored one for that
call only, which is how every tool evaluates a model at other parameter values.

In [ ]:
plant = MassSpringDamper()
print(plant)

u = np.array([0.0])
plant.f(plant.x0, u), plant.f(plant.x0, u, params={"m": 2.0, "k": 4.0, "c": 0.4})

Every tool takes the system as it is. `compute_trajectory` integrates `f` from `x0` and
returns a `Trajectory`, the sampled `(t, x, u, y)`; the system also keeps it for the plot.

In [ ]:
traj = plant.compute_trajectory(tf=10.0)

In [ ]:
plant.plot_trajectory()

## 2. Systems compose into a `DiagramSystem`

`controller @ plant` closes a feedback loop. The result is a `DiagramSystem`, itself a
`DynamicSystem` whose state stacks the blocks' states: every tool that works on a leaf works
on the loop. A PID reads an error, so `@` inserts the error block $e = r - y$.
`closed_loop` is the same operator with options; here it drives the reference with a step.

In [ ]:
from minilink import PID, Step, closed_loop

controller = PID(Kp=10.0, Ki=4.0, Kd=2.0)
loop = closed_loop(controller, plant, reference=Step(final_value=0.5, step_time=2.0))
print(loop)

traj_loop = loop.compute_trajectory(tf=10.0)

In [ ]:
loop.plot_trajectory()

### Wiring by hand

`@` is a shortcut for explicit wiring. A `DiagramSystem` holds named blocks
(`add_subsystem`) and connections from an output port to an input port (`connect`); a
boundary output is exposed with `connect_new_output_port`. The hand-wired loop below is the
same system as `loop`, and its trajectory matches.

In [ ]:
from minilink import DiagramSystem
from minilink.blocks import Error

diagram = DiagramSystem()
diagram.add_subsystem(Step(final_value=0.5, step_time=2.0), "reference")
diagram.add_subsystem(Error(), "error")
diagram.add_subsystem(controller, "controller")
diagram.add_subsystem(plant, "plant")

diagram.connect("reference", "y", "error", "+")
diagram.connect("plant", "y", "error", "-")
diagram.connect("error", "e", "controller", "e")
diagram.connect("controller", "u", "plant", "u")
diagram.connect_new_output_port("plant", "y", "y")

traj_diagram = diagram.compute_trajectory(tf=10.0)
np.allclose(traj_diagram.x, traj_loop.x)

## 3. Discrete time: `StepSystem`

A `StepSystem` is the sibling of `DynamicSystem` on an integer tick $k$. Its equation is a
next-state map instead of a rate:

$$x_{k+1} = \mathrm{step}(x_k, u_k, k; p), \qquad y_k = h(x_k, u_k, k; p)$$

`compute_rollout` iterates it for `n_steps` ticks and returns a `Trajectory` indexed by
tick. An accumulator, $x_{k+1} = x_k + u_k$:

In [ ]:
from minilink import StepSystem


class Accumulator(StepSystem):
    def __init__(self):
        super().__init__(1, input_dim=1, output_dim=1)

    def step(self, x, u, k=0, params=None):
        x_next = x + u

        return x_next


accumulator = Accumulator()
rollout = accumulator.compute_rollout(n_steps=10, u=np.array([1.0]))
rollout.x

Step systems compose among themselves with the same operators: `@` and `>>` between two
step systems give a `StepDiagramSystem`. A proportional law on the accumulator, in unity
feedback, converges to the reference geometrically.

In [ ]:
from minilink import P

digital_loop = P(0.3, ports="reference") @ Accumulator()
print(digital_loop)

rollout_loop = digital_loop.compute_rollout(n_steps=20, u=np.array([1.0]))

In [ ]:
digital_loop.plot_rollout()

## 4. A clock: `SampledSystem`

A clock is applied to a block from outside; it is never written into `f`. Both ways return
a `SampledSystem`, a step system whose tick lasts `sample_period` seconds and whose input is
held over each period. They differ in what the step means:

- `block % dt` is a **law** on a computer: the difference equation $x_{k+1} = x_k + \Delta t\, f(x_k, u_k)$, one Euler step per tick. This is how a controller goes digital.
- `discretize(block, dt)` (or `block.discretize(dt)`) is a **model**: the plant integrated over the period with `solver` and `substeps` (`"exact"` uses the matrix exponential on a linear model). This is what the discrete-time analysis and planning tools consume.

Both are views of the source: they read its live `params`.

In [ ]:
from minilink import discretize

law = plant % 0.1
model = discretize(plant, 0.1, solver="rk4", substeps=4)

x_next_law = law.step(plant.x0, u, 0)
x_next_model = model.step(plant.x0, u, 0)
law.kind, x_next_law, model.kind, x_next_model, model.params is plant.params

## 5. A digital controller on a continuous plant: `SampledDiagramSystem`

`(controller % dt) @ plant` is the sampled-data loop: the controller ticks every `dt`
seconds, the plant output is sampled at each tick, and the command is held while the plant
integrates in continuous time between ticks. The result is a `SampledDiagramSystem`, a step
diagram, so it runs with the step verbs; `compute_trajectory(tf, u)` runs it in seconds
with the loop input `u` (here the reference $r$) held.

In [ ]:
digital_controller = controller % 0.05
sampled_loop = digital_controller @ plant
print(sampled_loop)

traj_sampled = sampled_loop.compute_trajectory(tf=10.0, u=np.array([0.5]))

In [ ]:
sampled_loop.plot_trajectory(signals=("r", "y", "u"))

## Summary

| Object | Time | Equation | Built by |
| --- | --- | --- | --- |
| `DynamicSystem` | continuous | $\dot x = f(x, u, t)$ | subclass, catalog |
| `DiagramSystem` | continuous | stacked $f$ of its blocks | `@`, `>>`, `+`, `closed_loop`, `connect` |
| `StepSystem` | tick $k$ | $x_{k+1} = \mathrm{step}(x_k, u_k, k)$ | subclass |
| `StepDiagramSystem` | tick $k$ | stacked steps | `@`, `>>` between step systems |
| `SampledSystem` | tick of `dt` seconds | a law (`%`) or a model (`discretize`) | `block % dt`, `discretize` |
| `SampledDiagramSystem` | ticks + continuous plant | sample, hold, integrate | `(law % dt) @ plant` |

Next, [`02_parameters`](02_parameters.ipynb): every number in these objects is a parameter.